# 03 — Exploração do arquivo fenotípico


Este notebook realiza a exploração inicial do arquivo fenotípico do ADHD-200.

### Legenda do diagnóstico (`DX`)

- `0` = controle típico
- `1` = TDAH combinado
- `2` = TDAH hiperativo-impulsivo
- `3` = TDAH desatento

Também serão verificadas as variáveis de controle de qualidade (`QC_Athena` e `QC_NIAK`), a distribuição dos participantes por site e algumas características fenotípicas.

> **Importante:** este notebook é exploratório. A definição definitiva da amostra utilizada no TCC será realizada considerando os critérios de diagnóstico, controle de qualidade e correspondência com os dados de rs-fMRI utilizados nas etapas seguintes.

---
**Revisão (correções confirmadas neste notebook)**
1. `DX` e `QC_Athena` vinham como texto por causa de valores `pending` (e similares); a comparação `isin([0,1,2,3])` descartava todos os participantes. Agora as colunas são convertidas com `pd.to_numeric(errors="coerce")`; o que não é número vira `NaN` e é contado à parte.
2. Valores `-999` (código de dado ausente do ADHD-200) em idade e QI são convertidos para `NaN` antes de qualquer estatística.
3. A célula "Fisher z e vetorização" que estava no fim deste notebook foi **removida**: usava variáveis que não existem aqui (`fc`, `df`) e aplicaria o Fisher z uma segunda vez. O Fisher z é aplicado **uma única vez, no notebook 07**.
4. A contagem de sites passa a ser impressa por duas vias (coluna `Site` do fenótipo e pastas do download). O fenótipo usa 8 códigos de site (Peking é um único código) e o download tem 9 pastas (Peking_1, Peking_2 e Peking_3; parece não haver pasta para o código 2, com 26 participantes, provavelmente Bradley: confira). Por isso o 07 e o 08 trabalham com 9 centros; o "7 sites" que aparecia numa saída antiga deste notebook era resíduo de outra execução.


In [1]:
from pathlib import Path
import sys
import pandas as pd



def encontrar_raiz_projeto():
    """Procura a pasta que contém o config.py."""
    
    candidatos = [Path.cwd(), *Path.cwd().parents]
    
    for caminho in candidatos:
        if (caminho / "config.py").exists():
            return caminho.resolve()
    
    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto "
        "TCC_TDAH."
    )


RAIZ = encontrar_raiz_projeto()

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import config

if not config.ARQ_PHENO.exists():
    raise FileNotFoundError(
        f"Arquivo de fenótipos não encontrado:\n"
        f"{config.ARQ_PHENO}"
    )

pheno = pd.read_csv(
    config.ARQ_PHENO,
    sep="\t"
)

# --- Limpeza dos códigos do ADHD-200 ---------------------------------------
# 1) Guardar o DX original para auditar o que será convertido em NaN.
pheno["DX_original"] = pheno["DX"].astype(str).str.strip()

# 2) DX e QC_Athena podem conter texto ("pending", "-999"): converter para número.
for coluna in ["DX", "QC_Athena", "QC_NIAK"]:
    if coluna in pheno.columns:
        pheno[coluna] = pd.to_numeric(pheno[coluna], errors="coerce")

# 3) -999 é código de dado ausente em idade/QI (e também pode aparecer em DX).
colunas_com_999 = [
    c for c in ["DX", "Age", "IQ Measure", "Verbal IQ", "Performance IQ", "Full4 IQ", "Full2 IQ"]
    if c in pheno.columns
]
for coluna in colunas_com_999:
    pheno[coluna] = pheno[coluna].replace(-999, float("nan"))

print("DX original (antes da conversão):")
print(pheno["DX_original"].value_counts(dropna=False).sort_index())
print("\nDX após conversão numérica (NaN = pending/ausente):")
print(pheno["DX"].value_counts(dropna=False).sort_index())


print("ARQUIVO FENOTÍPICO — ADHD-200")


print(f"\nArquivo:")
print(f"  {config.ARQ_PHENO}")

print(f"\nDimensão:")
print(f"  {pheno.shape[0]:,} participantes × "
      f"{pheno.shape[1]:,} variáveis")

print("\nPrimeiras linhas:")
display(pheno.head())

print("\nColunas disponíveis:")
print(pheno.columns.tolist())

DX original (antes da conversão):
DX_original
0          585
1          212
2           13
3          137
pending     26
Name: count, dtype: int64

DX após conversão numérica (NaN = pending/ausente):
DX
0.0    585
1.0    212
2.0     13
3.0    137
NaN     26
Name: count, dtype: int64
ARQUIVO FENOTÍPICO — ADHD-200

Arquivo:
  /home/maria/TCC_TDAH/data/raw/adhd200_preprocessed_phenotypics.tsv

Dimensão:
  973 participantes × 20 variáveis

Primeiras linhas:


,ScanDir ID,Site,Gender,Age,Handedness,DX,Secondary Dx,ADHD Measure,ADHD Index,Inattentive,Hyper/Impulsive,IQ Measure,Verbal IQ,Performance IQ,Full2 IQ,Full4 IQ,Med Status,QC_Athena,QC_NIAK,DX_original
0,2371032,3,0.0,10.73,1,0.0,NaN,2,47,55,43,1.0,121.0,119.0,NaN,122.0,1,1.0,1.0,0
1,2026113,3,0.0,12.99,1,1.0,NaN,2,90,89,78,1.0,122.0,108.0,NaN,106.0,1,1.0,1.0,1
2,3434578,3,0.0,8.12,1,0.0,NaN,2,42,42,43,1.0,85.0,98.0,NaN,89.0,1,1.0,1.0,0
3,8628223,3,0.0,10.81,1,0.0,Simple phobia,2,42,49,49,1.0,85.0,86.0,NaN,97.0,1,1.0,1.0,0
4,1623716,3,0.0,12.65,1,1.0,NaN,2,87,90,90,1.0,89.0,88.0,NaN,89.0,1,1.0,1.0,1



Colunas disponíveis:
['ScanDir ID', 'Site', 'Gender', 'Age', 'Handedness', 'DX', 'Secondary Dx', 'ADHD Measure', 'ADHD Index', 'Inattentive', 'Hyper/Impulsive', 'IQ Measure', 'Verbal IQ', 'Performance IQ', 'Full2 IQ', 'Full4 IQ', 'Med Status', 'QC_Athena', 'QC_NIAK', 'DX_original']


In [2]:
# Distribuição do diagnóstico e controle de qualidade



print("DISTRIBUIÇÃO DO DIAGNÓSTICO")


if "DX" not in pheno.columns:
    raise KeyError("A coluna 'DX' não foi encontrada.")

print(pheno["DX"].value_counts(dropna=False).sort_index())



print("CONTROLE DE QUALIDADE — ATHENA")


if "QC_Athena" not in pheno.columns:
    raise KeyError("A coluna 'QC_Athena' não foi encontrada.")

print(pheno["QC_Athena"].value_counts(dropna=False).sort_index())



print("CONTROLE DE QUALIDADE — NIAK")


if "QC_NIAK" not in pheno.columns:
    print("Coluna 'QC_NIAK' não encontrada no arquivo.")
else:
    print(
        pheno["QC_NIAK"]
        .value_counts(dropna=False)
        .sort_index()
    )



print("INTERPRETAÇÃO DO DX")


print("0 = Controle")
print("1 = TDAH combinado")
print("2 = TDAH hiperativo-impulsivo")
print("3 = TDAH desatento")

DISTRIBUIÇÃO DO DIAGNÓSTICO
DX
0.0    585
1.0    212
2.0     13
3.0    137
NaN     26
Name: count, dtype: int64
CONTROLE DE QUALIDADE — ATHENA
QC_Athena
0.0     68
1.0    897
NaN      8
Name: count, dtype: int64
CONTROLE DE QUALIDADE — NIAK
QC_NIAK
0.0     50
1.0    915
NaN      8
Name: count, dtype: int64
INTERPRETAÇÃO DO DX
0 = Controle
1 = TDAH combinado
2 = TDAH hiperativo-impulsivo
3 = TDAH desatento


In [3]:
# Relação entre diagnóstico e QC_Athena


print("DIAGNÓSTICO × QC_ATHENA")


tabela_dx_qc = pd.crosstab(
    pheno["DX"],
    pheno["QC_Athena"],
    dropna=False
)

display(tabela_dx_qc)

DIAGNÓSTICO × QC_ATHENA


QC_Athena,0.0,1.0,NaN
DX,,,
0.0,35,547,3
1.0,18,190,4
2.0,1,12,0
3.0,13,123,1
NaN,1,25,0


In [4]:
# Distribuição após QC_Athena



print("PARTICIPANTES APROVADOS NO QC_ATHENA")


pheno_qc = pheno[
    pheno["QC_Athena"] == 1
].copy()

print(
    f"Participantes aprovados no QC_Athena: "
    f"{len(pheno_qc):,}"
)

print("\nDistribuição do DX após QC_Athena:")

print(
    pheno_qc["DX"]
    .value_counts(dropna=False)
    .sort_index()
)

PARTICIPANTES APROVADOS NO QC_ATHENA
Participantes aprovados no QC_Athena: 897

Distribuição do DX após QC_Athena:
DX
0.0    547
1.0    190
2.0     12
3.0    123
NaN     25
Name: count, dtype: int64


In [5]:
# Distribuição por site


print("PARTICIPANTES POR SITE")


if "Site" in pheno.columns:
    participantes_site = (
        pheno["Site"]
        .value_counts(dropna=False)
        .sort_index()
    )

    display(participantes_site)
    print(f"Sites distintos na coluna 'Site' do fenótipo: {pheno['Site'].nunique()}")
    print("Compare com a pasta de download (pastas por site) e com o notebook 04: "
          "lá aparecem 9 pastas (Peking_1/2/3 separados). "
          "PENDENTE DE VERIFICAÇÃO: se a diferença 7 x 9 é só o Peking.")

else:
    print("A coluna 'Site' não foi encontrada.")


# Características fenotípicas



print("CARACTERÍSTICAS FENOTÍPICAS")


colunas_numericas = [
    coluna
    for coluna in [
        "Age",
        "IQ Measure",
        "Verbal IQ",
        "Performance IQ",
        "Full4 IQ"
    ]
    if coluna in pheno.columns
]

if colunas_numericas:
    display(
        pheno[colunas_numericas].describe()
    )

else:
    print(
        "Nenhuma das variáveis fenotípicas esperadas "
        "foi encontrada."
    )

PARTICIPANTES POR SITE


Site
1    245
2     26
3     94
4     73
5    263
6    113
7     98
8     61
Name: count, dtype: int64

Sites distintos na coluna 'Site' do fenótipo: 8
Compare com a pasta de download (pastas por site) e com o notebook 04: lá aparecem 9 pastas (Peking_1/2/3 separados). PENDENTE DE VERIFICAÇÃO: se a diferença 7 x 9 é só o Peking.
CARACTERÍSTICAS FENOTÍPICAS


,Age,IQ Measure,Verbal IQ,Performance IQ,Full4 IQ
count,973.000000,898.000000,672.000000,672.000000,887.000000
mean,12.034872,2.387528,112.488095,106.531250,111.177903
std,3.328270,0.866277,14.461517,14.486097,13.844615
min,7.090000,1.000000,65.000000,54.000000,73.000000
25%,9.330000,2.000000,103.750000,97.000000,102.000000
50%,11.420000,2.000000,112.000000,106.000000,111.000000
75%,14.000000,3.000000,122.000000,117.000000,121.000000
max,26.310000,5.000000,158.000000,143.000000,153.000000


In [6]:
# Criar variável binária para classificação

print("=" * 60)
print("CLASSIFICAÇÃO BINÁRIA — CONTROLE × TDAH")
print("=" * 60)

# Selecionar somente participantes aprovados no QC Athena


pheno_qc = pheno[
    pheno["QC_Athena"] == 1
].copy()

print(
    f"\nParticipantes aprovados no QC_Athena: "
    f"{len(pheno_qc):,}"
)

# Remover diagnósticos pendentes


pheno_qc = pheno_qc[
    pheno_qc["DX"].isin([0, 1, 2, 3])
].copy()

print(
    f"Participantes com diagnóstico válido: "
    f"{len(pheno_qc):,}"
)

# ---------------------------------------------------------
# classificação binária
# ---------------------------------------------------------
#
# DX = 0 → Controle → target 0
# DX = 1 → TDAH     → target 1
# DX = 2 → TDAH     → target 1
# DX = 3 → TDAH     → target 1
#
# DX = pending foi excluído anteriormente.
# ---------------------------------------------------------

pheno_qc["target"] = pheno_qc["DX"].apply(
    lambda x: 0 if x == 0 else 1
)

print("\nDistribuição da variável target:")

contagem_target = (
    pheno_qc["target"]
    .value_counts()
    .sort_index()
)

display(contagem_target)

print("\nLegenda:")
print("0 = Controle")
print("1 = TDAH")

# Sites que só têm uma classe (confusão site × diagnóstico)
if "Site" in pheno_qc.columns:
    tab_site_target = pd.crosstab(pheno_qc["Site"], pheno_qc["target"])
    tab_site_target.columns = ["Controle" if c == 0 else "TDAH" for c in tab_site_target.columns]
    display(tab_site_target)
    so_um = tab_site_target[(tab_site_target == 0).any(axis=1)]
    print("Sites com apenas uma classe nesta etapa (ainda antes de cruzar com os arquivos rs-fMRI):")
    display(so_um)


CLASSIFICAÇÃO BINÁRIA — CONTROLE × TDAH

Participantes aprovados no QC_Athena: 897
Participantes com diagnóstico válido: 872

Distribuição da variável target:


target
0    547
1    325
Name: count, dtype: int64


Legenda:
0 = Controle
1 = TDAH


,Controle,TDAH
Site,,
1,141,99
3,64,24
4,35,30
5,97,128
6,64,40
7,92,4
8,54,0


Sites com apenas uma classe nesta etapa (ainda antes de cruzar com os arquivos rs-fMRI):


,Controle,TDAH
Site,,
8,54,0


In [7]:
# Resumo da exploração


print("RESUMO — ADHD-200")


print(f"\nParticipantes no arquivo fenotípico: {len(pheno):,}")

print(
    f"Participantes aprovados no QC_Athena: "
    f"{len(pheno_qc):,}"
)

print("\nDistribuição final após QC:")

contagem_target = (
    pheno_qc["target"]
    .value_counts()
    .sort_index()
)

for classe, quantidade in contagem_target.items():

    nome = "Controle" if classe == 0 else "TDAH"

    percentual = (
        quantidade / len(pheno_qc) * 100
    )

    print(
        f"  {nome}: {quantidade:,} "
        f"({percentual:.2f}%)"
    )


print("✓ EXPLORAÇÃO CONCLUÍDA")



RESUMO — ADHD-200

Participantes no arquivo fenotípico: 973
Participantes aprovados no QC_Athena: 872

Distribuição final após QC:
  Controle: 547 (62.73%)
  TDAH: 325 (37.27%)
✓ EXPLORAÇÃO CONCLUÍDA
